# 3.2 — Inverse Kinematics

Solve for joint angles that place a site (e.g. a fingertip) at a target position, using MuJoCo's Jacobians. The solver lives in `files/3.2/inverse_kinematics.py`; the MyoChallenge 2025 demos are in `files/3.2/mc25/`.

**Prerequisites:** Completed notebook 1.1 · `myosuite` installed

The solver module `files/3.2/inverse_kinematics.py` is importable: it exposes the model, the `mink` configuration and the tasks. Here we move the target (a mocap body) along a circle, let `mink` chase it, and render the arm inline.

In [ ]:
import sys
from pathlib import Path

import mink
import mujoco
import numpy as np

from myosuite.utils.video_io import show_video, write_video

ik_dir = next(p for p in (Path("files/3.2"), Path("tutorials/files/3.2")) if p.exists())
sys.path.insert(0, str(ik_dir.resolve()))
import inverse_kinematics as ik  # noqa: E402  (builds the arm + mocap target model)

model, data, configuration = ik.model, ik.data, ik.configuration
configuration.update(data.qpos)
ik.posture_task.set_target_from_configuration(configuration)  # keep the rest of the arm near its start pose
mujoco.mj_forward(model, data)
mink.move_mocap_to_frame(model, data, "target", "S_grasp", "site")  # target starts at the hand
start = data.mocap_pos[0].copy()
print("start position of the hand:", start.round(3))

Track a circular target path (a circle above the hanging hand, which the arm can reach): each frame moves the target, runs a few IK iterations, and renders the result.

In [ ]:
N_FRAMES, RADIUS = 120, 0.08
center = start + np.array([0.0, 0.0, 0.1])  # the arm hangs down, so circle above the start pose
renderer = mujoco.Renderer(model, height=360, width=480)
cam = mujoco.MjvCamera()
mujoco.mjv_defaultFreeCamera(model, cam)
cam.lookat[:] = center
cam.distance, cam.azimuth, cam.elevation = 1.2, 150, -20

frames, errors = [], []
for k in range(N_FRAMES):
    angle = 2 * np.pi * k / N_FRAMES
    data.mocap_pos[0] = center + RADIUS * np.array([0.0, np.sin(angle), -np.cos(angle)])
    ik.end_effector_task.set_target(mink.SE3.from_mocap_name(model, data, "target"))
    for _ in range(30):
        vel = mink.solve_ik(configuration, ik.tasks, 0.01, ik.solver, 1e-3)
        configuration.integrate_inplace(vel, 0.01)
    err = ik.end_effector_task.compute_error(configuration)
    errors.append(np.linalg.norm(err[:3]))
    data.qpos[:] = configuration.q
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, camera=cam)
    frames.append(renderer.render())
renderer.close()
print(f"mean position error: {np.mean(errors):.4f} m")

In [ ]:
Path("videos").mkdir(exist_ok=True)
write_video("videos/inverse_kinematics.mp4", np.asarray(frames), fps=30, outputdict={"-pix_fmt": "yuv420p"})
show_video("videos/inverse_kinematics.mp4")

Set `MUJOCO_GL=egl` (or `osmesa`) for headless machines. For an interactive viewer run `python files/3.2/inverse_kinematics.py` (`mjpython` on macOS).